# Forward Modelling Registered - NEMES 2026 workshop

Does forward modelling using the **Colin27 template**.

With the addition of registered geometry from previous step.

1. Load one SNIRF file
2. Apply saved geometry
3. Register optodes to Colin27 scalp
4. Compute forward model sensitivity matrix via nirfaster (i.e. channel space -> surface space matrix)

This is adapted from Cedalion tutorial notebooks. For details on each step, see Cedalion docs:

- [Tutorial 1 – Heads and Forward Models](https://doc.ibs.tu-berlin.de/cedalion/doc/dev/examples/tutorial/1_heads_and_fwm.html)


In [ ]:
import glob
import os

import cedalion
import cedalion.io.snirf
import cedalion.nirs.cw
import cedalion.vis.anatomy
import cedalion.vis.blocks as vbx
from cedalion import units

import pyvista as pv
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

np.set_printoptions(suppress=True)

# What counts as a short channel?
DIST_THRESHOLD = 1.5 * units.cm 

snirf_file = "../data/example.snirf"
tsv_filename = "../data/registration/geometry.tsv"

fwm_dir = Path("../data/forward_model_registered")
fwm_dir.mkdir(parents=True, exist_ok=True)

# Load data

We will load the SNIRF file as previously and plot the montage.

In [ ]:
rec = cedalion.io.snirf.read_snirf(snirf_file)[0]
geo3d_orig = rec.geo3d

amp = rec.get_timeseries()
print("Amplitude shape:", dict(amp.sizes))
print("Wavelengths (nm):", amp.wavelength.values)
print("geo3d_orig labels:", geo3d_orig.label.values)

# Plot montage
cedalion.vis.anatomy.montage.plot_montage3D(rec['amp'], geo3d_orig)

# Import registered geometry

In [ ]:
# Import registered geometry
geo3d_loaded = cedalion.io.probe_geometry.load_tsv(tsv_filename)
print("geo3d_loaded labels:", geo3d_loaded.label.values)

# Compare montages

In [ ]:
f,ax = plt.subplots(1,2, figsize=(12,6))
cedalion.vis.anatomy.scalp_plot(
    rec["amp"],
    geo3d_orig,
    cedalion.nirs.channel_distances(rec["amp"], geo3d_orig),
    ax=ax[0],
    optode_labels=True,
    cb_label="channel dist. / mm",
    cmap="plasma",
    vmin=25,
    vmax=42,
)
ax[0].set_title("montage from snirf file")
cedalion.vis.anatomy.scalp_plot(
    rec["amp"],
    geo3d_loaded,
    cedalion.nirs.channel_distances(rec["amp"], geo3d_loaded),
    ax=ax[1],
    optode_labels=True,
    cb_label="channel dist. / mm",
    cmap="plasma",
    vmin=25,
    vmax=42,
)
ax[1].set_title("montage from photogrammetric scan")
plt.tight_layout()

# Get head model

We will snap the loaded geometry `geo3d_loaded` to the head model instead of the original montage.

In [ ]:
head_ijk = cedalion.dot.get_standard_headmodel("colin27")

print("Scalp vertices:", head_ijk.scalp)
print("Brain vertices:", head_ijk.brain)
print("Landmarks:", head_ijk.landmarks.label.values)

# First rename landmarks in registered geometry to head landmarks
geo3d_loaded = geo3d_loaded.points.rename({"Lpa": "LPA",
                                       "Rpa": "RPA"})

# Register optodes: snap to Colin27 scalp using the cranial landmarks
geo3d_snapped_ijk, alignment_details = head_ijk.align_and_relax_to_scalp(
    geo3d_loaded, amp
)
print("Snapped geo3d dims:", dict(geo3d_snapped_ijk.sizes))

head_ras = head_ijk.apply_transform(head_ijk.t_ijk2ras)
display(head_ras)

# Let's plot the montage on the head
plt = pv.Plotter()
vbx.plot_surface(plt, head_ijk.scalp, color="#4fce64", opacity=.1)
vbx.plot_labeled_points(plt, geo3d_snapped_ijk)
vbx.plot_labeled_points(
    plt, head_ijk.landmarks.sel(label=["Nz", "Iz", "Cz", "LPA", "RPA"]), color="y"
)
plt.show()

# Compute forward model

In [ ]:
fluence_file     = f"{fwm_dir}/colin27_fluence.h5"
sensitivity_file = f"{fwm_dir}/colin27_Adot.h5"

if not os.path.exists(sensitivity_file):
    print("Computing forward model...")
    meas_list = rec._measurement_lists["amp"]
    fwm = cedalion.dot.ForwardModel(head_ijk, geo3d_snapped_ijk, meas_list)

    if not os.path.exists(fluence_file):
        fwm.compute_fluence_nirfaster(fluence_file)

    fwm.compute_sensitivity(fluence_file, sensitivity_file)
    print(f"Saved to {sensitivity_file}")
else:
    print(f"Loading cached sensitivity: {sensitivity_file}")

Adot = cedalion.io.load_Adot(sensitivity_file)
print("Adot dims:", dict(Adot.sizes))

# Visualize sensitivity

This time, we will have the sensitivity of the actual montage on the head, instead of the original version defined in software.

In [ ]:
# Split long / short channels
amp_long, amp_short = cedalion.nirs.split_long_short_channels(
    amp, rec.geo3d, DIST_THRESHOLD
)
print(f"Long channels: {amp_long.sizes['channel']}")
print(f"Short channels: {amp_short.sizes['channel']}")

# Restrict Adot to long channels
shared_channels = np.intersect1d(Adot.channel.values, amp_long.channel.values)
Adot_long = Adot.sel(channel=shared_channels)

print(f"Long channels in Adot: {len(shared_channels)}")

In [ ]:
# Select only a subset of labeled points to plot
# Here we select sources and detectors via their labels that start with S or D
# I.e. skip landmarks and channel centers
geo3d_plot_ijk = geo3d_snapped_ijk.sel(
    label=geo3d_snapped_ijk.label.str.contains("S|D")
)

plotter = cedalion.vis.anatomy.sensitivity_matrix.Main(
    sensitivity=Adot_long,
    brain_surface=head_ijk.brain,
    head_surface=head_ijk.scalp,
    labeled_points= geo3d_plot_ijk,
)
plotter.plot(high_th=0, low_th=-2)
plotter.plt.show()